# 13 · DBU Usage Analysis — System Tables
**Databricks Unit (DBU) utilization tracking using `system.billing.usage`**

| Cell | What it shows |
|---|---|
| S1 | Raw system table preview |
| S2 | Daily Jobs vs Ad-hoc DBU split |
| S3 | This month KPI summary |
| S4 | VStone pipeline-specific usage |
| S5 | Weekly trend (last 90 days) |
| S6 | SKU breakdown (Serverless vs Classic) |
| S7 | Peak usage days |

> **Prerequisite:** Admin must grant `SELECT ON system.billing.usage` to your user.
> Run: `GRANT SELECT ON system.billing.usage TO <your_user>`

## S1 — System Table Preview
Quick look at available columns and recent raw rows.

In [0]:
-- Test 1: Check what system tables you CAN access
SHOW TABLES IN system.billing

In [0]:
-- Daily DBU usage with actual USD cost
SELECT
  DATE(u.usage_date)                                    AS usage_day,
  u.usage_type,
  u.sku_name,
  ROUND(SUM(u.usage_quantity), 2)                       AS total_dbu,
  ROUND(AVG(p.pricing.default), 4)                      AS price_per_dbu,
  ROUND(SUM(u.usage_quantity) * AVG(p.pricing.default), 2) AS estimated_cost_usd
FROM system.billing.usage u
LEFT JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
WHERE u.usage_date >= DATEADD(DAY, -30, CURRENT_DATE)
GROUP BY DATE(u.usage_date), u.usage_type, u.sku_name
ORDER BY usage_day DESC, estimated_cost_usd DESC

In [0]:
-- What each compute type costs per DBU
SELECT
  sku_name,
  pricing.default   AS price_per_dbu_usd,
  currency_code,
  pricing_unit,
  effective_from
FROM system.billing.list_prices
ORDER BY price_per_dbu_usd DESC

In [0]:
GRANT SELECT ON system.billing.usage TO `akashmishraa202@gmail.com`

In [0]:
%sql
-- Preview system.billing.usage schema and last 10 rows
SELECT
  usage_date,
  workspace_id,
  sku_name,
  usage_type,
  usage_unit,
  usage_quantity,
  usage_metadata.job_id    AS job_id,
  usage_metadata.job_name  AS job_name,
  usage_metadata.notebook_id   AS notebook_id,
  usage_metadata.notebook_path AS notebook_path
FROM system.billing.usage
ORDER BY usage_date DESC
LIMIT 10

## S2 — Daily DBU: Jobs vs Ad-hoc Queries
**Core metric** — shows exactly how many DBUs were consumed each day,
split between automated jobs and interactive/ad-hoc notebook queries.

| Column | Meaning |
|---|---|
| `jobs_dbu` | DBUs from scheduled/DLT pipeline runs |
| `adhoc_dbu` | DBUs from manual notebook/SQL executions |
| `jobs_pct` | % of day's DBUs from jobs |

In [0]:
%sql
-- Daily DBU split: Jobs vs Interactive (Ad-hoc) — last 30 days
SELECT
  DATE(usage_date)                                               AS usage_day,

  -- Jobs DBU (DLT pipelines, scheduled jobs)
  ROUND(SUM(CASE WHEN usage_type = 'JOBS'
                 THEN usage_quantity ELSE 0 END), 2)            AS jobs_dbu,

  -- Ad-hoc / Interactive DBU (notebook runs, SQL queries)
  ROUND(SUM(CASE WHEN usage_type = 'INTERACTIVE'
                 THEN usage_quantity ELSE 0 END), 2)            AS adhoc_dbu,

  -- Serverless DBU (if using serverless compute)
  ROUND(SUM(CASE WHEN sku_name LIKE '%SERVERLESS%'
                 THEN usage_quantity ELSE 0 END), 2)            AS serverless_dbu,

  -- Total DBU for the day
  ROUND(SUM(usage_quantity), 2)                                 AS total_dbu,

  -- Jobs share
  ROUND(
    SUM(CASE WHEN usage_type = 'JOBS'
        THEN usage_quantity ELSE 0 END)
    / NULLIF(SUM(usage_quantity), 0) * 100, 1
  )                                                             AS jobs_pct,

  -- Ad-hoc share
  ROUND(
    SUM(CASE WHEN usage_type = 'INTERACTIVE'
        THEN usage_quantity ELSE 0 END)
    / NULLIF(SUM(usage_quantity), 0) * 100, 1
  )                                                             AS adhoc_pct,

  COUNT(*)                                                      AS billing_records

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -30, CURRENT_DATE)
GROUP BY DATE(usage_date)
ORDER BY usage_day DESC

## S3 — This Month KPI Summary
Single-row summary of the current month's DBU consumption.

In [0]:
%sql
-- Current month DBU summary — KPI counters
SELECT
  DATE_TRUNC('month', CURRENT_DATE)                             AS month,

  -- Total DBU this month
  ROUND(SUM(usage_quantity), 1)                                 AS total_dbu_this_month,

  -- By type
  ROUND(SUM(CASE WHEN usage_type = 'JOBS'
                 THEN usage_quantity ELSE 0 END), 1)            AS jobs_dbu,

  ROUND(SUM(CASE WHEN usage_type = 'INTERACTIVE'
                 THEN usage_quantity ELSE 0 END), 1)            AS adhoc_dbu,

  ROUND(SUM(CASE WHEN sku_name LIKE '%SERVERLESS%'
                 THEN usage_quantity ELSE 0 END), 1)            AS serverless_dbu,

  -- Active days
  COUNT(DISTINCT DATE(usage_date))                              AS active_days,

  -- Daily average
  ROUND(SUM(usage_quantity)
        / NULLIF(COUNT(DISTINCT DATE(usage_date)), 0), 2)       AS avg_dbu_per_active_day,

  -- Peak single day
  ROUND(MAX(daily_total), 2)                                    AS peak_dbu_day

FROM (
  SELECT
    usage_date,
    usage_type,
    sku_name,
    usage_quantity,
    SUM(usage_quantity) OVER (PARTITION BY DATE(usage_date))    AS daily_total
  FROM system.billing.usage
  WHERE DATE_TRUNC('month', usage_date) = DATE_TRUNC('month', CURRENT_DATE)
) t

## S4 — VStone Pipeline DBU Usage
Filters to **your specific DLT pipelines** (Bronze → Silver → Gold).
Shows which pipeline consumes the most DBUs.

> If `job_name` is NULL, your pipelines may not have names set in the job config.

In [0]:
%sql
-- VStone pipeline-specific DBU usage — last 30 days
SELECT
  DATE(usage_date)                                              AS usage_day,
  usage_metadata.job_id                                         AS job_id,
  COALESCE(
    usage_metadata.job_name,
    CONCAT('job_', usage_metadata.job_id)
  )                                                             AS pipeline_name,
  sku_name,
  usage_type,
  ROUND(SUM(usage_quantity), 3)                                 AS total_dbu,
  COUNT(*)                                                      AS run_count

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -30, CURRENT_DATE)
  AND usage_type = 'JOBS'
  AND (
    LOWER(usage_metadata.job_name) LIKE '%vstone%'
    OR LOWER(usage_metadata.job_name) LIKE '%bronze%'
    OR LOWER(usage_metadata.job_name) LIKE '%silver%'
    OR LOWER(usage_metadata.job_name) LIKE '%gold%'
    OR LOWER(usage_metadata.job_name) LIKE '%dlt%'
    OR LOWER(usage_metadata.job_name) LIKE '%chunk%'
    OR usage_metadata.job_id IS NOT NULL   -- fallback: show all jobs
  )
GROUP BY
  DATE(usage_date),
  usage_metadata.job_id,
  usage_metadata.job_name,
  sku_name,
  usage_type
ORDER BY usage_day DESC, total_dbu DESC

## S5 — Weekly DBU Trend (Last 90 Days)
Shows week-over-week DBU patterns — useful for spotting unusual spikes.

In [0]:
%sql
-- Weekly DBU trend — last 90 days
SELECT
  DATE_TRUNC('week', usage_date)                                AS week_start,
  DATE_ADD(DATE_TRUNC('week', usage_date), 6)                   AS week_end,

  -- By type
  ROUND(SUM(CASE WHEN usage_type = 'JOBS'
                 THEN usage_quantity ELSE 0 END), 2)            AS jobs_dbu,
  ROUND(SUM(CASE WHEN usage_type = 'INTERACTIVE'
                 THEN usage_quantity ELSE 0 END), 2)            AS adhoc_dbu,

  -- Totals
  ROUND(SUM(usage_quantity), 2)                                 AS total_dbu,
  ROUND(AVG(usage_quantity), 2)                                 AS avg_daily_dbu,
  ROUND(MAX(usage_quantity), 2)                                 AS peak_single_day_dbu,
  COUNT(DISTINCT DATE(usage_date))                              AS active_days

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -90, CURRENT_DATE)
GROUP BY DATE_TRUNC('week', usage_date)
ORDER BY week_start DESC

## S6 — DBU by SKU / Compute Type
Shows **which compute type** is consuming DBUs:
- `STANDARD_ALL_PURPOSE_COMPUTE` — interactive clusters
- `STANDARD_JOBS_COMPUTE` — job clusters
- `PREMIUM_ALL_PURPOSE_COMPUTE` — premium tier
- `SERVERLESS_REAL_TIME_INFERENCE` — serverless SQL

In [0]:
%sql
-- DBU consumption by SKU type — last 30 days
SELECT
  sku_name,
  usage_type,

  ROUND(SUM(usage_quantity), 2)                                 AS total_dbu,
  ROUND(AVG(usage_quantity), 3)                                 AS avg_dbu_per_record,
  COUNT(DISTINCT DATE(usage_date))                              AS days_used,
  COUNT(*)                                                      AS billing_records,

  -- Share of total
  ROUND(
    SUM(usage_quantity)
    / SUM(SUM(usage_quantity)) OVER () * 100, 1
  )                                                             AS pct_of_total_dbu

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -30, CURRENT_DATE)
GROUP BY sku_name, usage_type
ORDER BY total_dbu DESC

## S7 — Peak Usage Days
Top 10 highest DBU consumption days — helps identify which pipeline runs caused spikes.

In [0]:
%sql
-- Top 10 peak DBU days with breakdown
SELECT
  DATE(usage_date)                                              AS usage_day,
  DAYOFWEEK(usage_date)                                         AS day_of_week,
  DATE_FORMAT(usage_date, 'EEEE')                               AS day_name,

  ROUND(SUM(usage_quantity), 2)                                 AS total_dbu,

  ROUND(SUM(CASE WHEN usage_type = 'JOBS'
                 THEN usage_quantity ELSE 0 END), 2)            AS jobs_dbu,

  ROUND(SUM(CASE WHEN usage_type = 'INTERACTIVE'
                 THEN usage_quantity ELSE 0 END), 2)            AS adhoc_dbu,

  -- What drove the spike
  COUNT(DISTINCT usage_metadata.job_id)                         AS distinct_jobs_ran,
  COUNT(*)                                                      AS billing_records

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -90, CURRENT_DATE)
GROUP BY DATE(usage_date), DAYOFWEEK(usage_date), DATE_FORMAT(usage_date, 'EEEE')
ORDER BY total_dbu DESC
LIMIT 10

## S8 — Ad-hoc Notebook DBU Breakdown
Shows which specific notebooks are consuming the most interactive DBUs.

In [0]:
%sql
-- Which notebooks consume the most ad-hoc DBUs
SELECT
  COALESCE(
    usage_metadata.notebook_path,
    CONCAT('notebook_', usage_metadata.notebook_id)
  )                                                             AS notebook_path,
  usage_metadata.notebook_id                                    AS notebook_id,

  ROUND(SUM(usage_quantity), 3)                                 AS total_dbu,
  COUNT(DISTINCT DATE(usage_date))                              AS days_used,
  COUNT(*)                                                      AS execution_records,
  MIN(DATE(usage_date))                                         AS first_used,
  MAX(DATE(usage_date))                                         AS last_used

FROM system.billing.usage
WHERE usage_date >= DATEADD(DAY, -30, CURRENT_DATE)
  AND usage_type = 'INTERACTIVE'
  AND usage_metadata.notebook_id IS NOT NULL
GROUP BY
  usage_metadata.notebook_path,
  usage_metadata.notebook_id
ORDER BY total_dbu DESC
LIMIT 20

## Summary

| Query | Purpose | Table Used |
|---|---|---|
| S1 | Raw preview of billing data | `system.billing.usage` |
| S2 | **Daily Jobs vs Ad-hoc split** ← main metric | `system.billing.usage` |
| S3 | This month KPI counters | `system.billing.usage` |
| S4 | VStone pipeline-specific DBU | `system.billing.usage` |
| S5 | Weekly trend — 90 day view | `system.billing.usage` |
| S6 | SKU / compute type breakdown | `system.billing.usage` |
| S7 | Peak usage days (top 10) | `system.billing.usage` |
| S8 | Notebook-level ad-hoc usage | `system.billing.usage` |

### Key Fields in `system.billing.usage`
| Field | Values |
|---|---|
| `usage_type` | `JOBS`, `INTERACTIVE` |
| `sku_name` | `STANDARD_ALL_PURPOSE_COMPUTE`, `STANDARD_JOBS_COMPUTE`, `SERVERLESS_*` |
| `usage_quantity` | DBU count consumed |
| `usage_metadata.job_name` | Name of the Databricks job |
| `usage_metadata.notebook_path` | Path of the notebook executed |

> **Tip:** If you see NULL job names in S4, go to **Workflows → your DLT pipeline → Settings**
> and add a proper name so it appears correctly in these queries.